# 05b · HEO audit: labels or model?

The teachers reach 0.88 fresh-burn IoU on PhiSat-2 and about 0.5 on HEO. Before the search, we need to know how much of that gap is the model and how much is the labels: NAS can only work on the first part.

Here both teachers, averaged, predict every HEO scene that has fresh burn, whole and at its own resolution (2.5 m), and the predictions are laid over the labels:

- **labelled fresh, teachers agree** (dark red), **labelled fresh, teachers say no** (amber), **teachers say fresh, label says no** (blue);
- per scene numbers, for each teacher and their average;
- what the amber and blue pixels look like in reflectance, next to the pixels where teachers and labels agree;
- how far the blue pixels are from a labelled burn: next to it is a disagreement about the edge, far from it is a separate object, a burn EFFIS missed or dark ground the model takes for one.

On the training scenes the teachers have seen these labels 40 times. What they still refuse to call fresh there is most likely not visibly burnt. The validation and test scenes are only looked at here; nothing is tuned to them.

In [ ]:
# [1] setup: both teachers
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                # the notebooks run from the repository root
import os, sys, json, glob, time, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from scipy import ndimage
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import torch
from torch.utils.data import DataLoader, Dataset
import burnscar.labels as L
import burnscar.dataset as D
import burnscar.train as T
for module in (L, D, T):
    importlib.reload(module)

cfg = json.load(open("data/tables/dataset_config.json"))
scenes = pd.read_csv(cfg["files"]["scenes"])
norm = json.load(open(cfg["files"]["normalisation"]))
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
amp_dtype, _ = T.amp_settings(device)
teachers = {}
for p in sorted(glob.glob("models/teacher/*/best.pt")):
    ck = torch.load(p, map_location=device, weights_only=False)
    m = T.build_unet(ck["settings"]["encoder"], in_channels=7, classes=T.N_CLASSES, pretrained=False).to(device)
    m.load_state_dict(ck["model"])
    name = ck["settings"].get("run", os.path.basename(os.path.dirname(p)))
    teachers[name] = m.eval()
    print(f"{name}: epoch {ck['epoch']}, validation score {ck['score']:.3f}")
OUT = "models/teacher/audit"
os.makedirs(OUT, exist_ok=True)

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
C_BLUE, C_ORANGE, C_GREY, INK2 = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
plt.rcParams.update({"axes.grid": True, "grid.color": "#e6e5e0", "axes.axisbelow": True,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
                     "axes.edgecolor": "#8a8984", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2})
# agreement map: 0 clear agreed, 1 fresh agreed, 2 labelled fresh but teachers say no, 3 teachers fresh but label no,
# 4 other labels (old, cloud, shadow, water), 5 left out, 6 no data
AGREE_NAMES = ["clear, agreed", "fresh, agreed", "labelled fresh, teachers no", "teachers fresh, label no",
               "other classes", "left out", "no data"]
AGREE_COLORS = ["#e9e8e4", L.V3_COLORS[L.FRESH], "#f2b134", C_BLUE, "#c9c7bf", L.V3_COLORS[L.IGNORE], "#ffffff"]
AGREE_CMAP = ListedColormap(AGREE_COLORS)
SPLIT_ORDER = {"val": 0, "test": 1, "train": 2}

In [ ]:
# [2] the HEO scenes with fresh burn: validation, test and training
heo = scenes[(scenes.source == "heo") & scenes.split.isin(list(SPLIT_ORDER)) & (scenes.fresh_px > 0)].copy()
heo["name"] = heo.scene.str[:5] + " " + heo.scene.str[17:32]
heo = heo.assign(o=heo.split.map(SPLIT_ORDER)).sort_values(["o", "fresh_px"], ascending=[True, False]).drop(columns="o")
heo[["name", "split", "level", "fresh_px", "old_px", "geo_rms_m", "group"]].reset_index(drop=True)

In [ ]:
# [3] what is computed for each scene (numpy only, nothing here touches the GPU)
def agreement(label, cls_map, raw):
    """Agreement map (codes in AGREE_NAMES) from training ids (-1..5), predicted classes and raw v3 mask values."""
    a = np.full(label.shape, 4, np.uint8)
    a[(label == 0) & (cls_map != 1)] = 0
    a[(label == 1) & (cls_map == 1)] = 1
    a[(label == 1) & (cls_map != 1)] = 2
    a[(label == 0) & (cls_map == 1)] = 3
    a[raw == L.IGNORE] = 5
    a[raw == L.NODATA] = 6
    return a


def fresh_scores(label, cls_map):
    """Fresh-burn precision, recall and IoU over labelled pixels (old, cloud, shadow, water count as not fresh)."""
    ok = label >= 0
    tp = int(((label == 1) & (cls_map == 1) & ok).sum())
    fp = int(((label != 1) & (cls_map == 1) & ok).sum())
    fn = int(((label == 1) & (cls_map != 1) & ok).sum())
    return dict(tp=tp, fp=fp, fn=fn, precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1),
                iou=tp / max(tp + fp + fn, 1))


def sample_groups(a, refl, rng, k=20000):
    """Reflectance (7 bands) of up to k random pixels of each agreement group 0-3."""
    rows = []
    for g in (0, 1, 2, 3):
        rr, cc = np.nonzero(a == g)
        if len(rr) == 0:
            continue
        pick = rng.choice(len(rr), min(k, len(rr)), replace=False)
        v = refl[:, rr[pick], cc[pick]].astype(np.float32).T
        rows.append(pd.DataFrame(v, columns=D.BAND_ORDER).assign(group=g))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def fp_distances(label, a, px_m):
    """Distance in metres from every 'teachers fresh, label no' pixel to the nearest labelled fresh pixel."""
    if not (a == 3).any():
        return np.zeros(0, np.float32)
    dist = ndimage.distance_transform_edt(label != 1).astype(np.float32) * px_m
    return dist[a == 3]


def false_colour(refl):
    """NIR, red, green stretched 2-98% over pixels with data; no data white. refl: (7, H, W)."""
    a = refl[[6, 2, 1]].astype(np.float32)
    valid = (a > 0).all(0)
    out = np.ones(a.shape[1:] + (3,), np.float32)
    for i in range(3):
        lo, hi = np.percentile(a[i][valid], (2, 98)) if valid.sum() > 10 else (0, 1)
        out[..., i] = np.clip((a[i] - lo) / max(hi - lo, 1e-6), 0, 1)
    out[~valid] = 1
    return out


def biggest_blobs(mask, n=2, min_px=200):
    """Centres (row, col) of the n largest connected blobs of a boolean mask."""
    lab, k = ndimage.label(mask)
    if k == 0:
        return []
    sizes = ndimage.sum(mask, lab, index=np.arange(1, k + 1))
    order = [i + 1 for i in np.argsort(sizes)[::-1][:n] if sizes[i] >= min_px]
    return [tuple(int(v) for v in ndimage.center_of_mass(lab == i)) for i in order]

In [ ]:
# [4] every scene, whole, at its own resolution: both teachers and their average (a few minutes on the GPU)
class GridWithRefl(Dataset):
    def __init__(self, fw):
        self.fw = fw
    def __len__(self):
        return len(self.fw)
    def __getitem__(self, i):
        x, y, refl, msk = self.fw.get(i, with_refl=True)
        return x, y, refl.astype(np.float16), msk, i


def predict_scene(r, batch=32, workers=4):
    boxes = D.grid_boxes(int(r.height), int(r.width), r.px_m, r.px_m, 256)
    fw = D.FixedWindows(pd.DataFrame([dict(scene=r.scene, r0=b[0], c0=b[1], n=b[2]) for b in boxes]), scenes, norm, size=256)
    H, W = int(r.height), int(r.width)
    label, raw = np.full((H, W), -1, np.int8), np.full((H, W), L.NODATA, np.uint8)
    refl = np.zeros((7, H, W), np.float16)
    names = list(teachers) + ["average"]
    cls = {k: np.zeros((H, W), np.uint8) for k in names}
    with torch.no_grad():
        for x, y, rf, msk, idx in DataLoader(GridWithRefl(fw), batch_size=batch, num_workers=workers):
            xg = x.to(device)
            with T.autocast(device, amp_dtype):
                ps = {k: m(xg).float().softmax(1) for k, m in teachers.items()}
            ps["average"] = sum(ps.values()) / len(teachers)
            pc = {k: p.argmax(1).to(torch.uint8).cpu().numpy() for k, p in ps.items()}
            for j, i in enumerate(idx.tolist()):
                r0, c0, n = boxes[i]
                label[r0:r0 + n, c0:c0 + n] = y[j].numpy()
                raw[r0:r0 + n, c0:c0 + n] = msk[j].numpy()
                refl[:, r0:r0 + n, c0:c0 + n] = rf[j].numpy()
                for k in names:
                    cls[k][r0:r0 + n, c0:c0 + n] = pc[k][j]
    return label, raw, refl, cls


rng = np.random.default_rng(0)
rows, samples, dists, views = [], [], {}, {}
t0 = time.time()
for r in heo.itertuples():
    label, raw, refl, cls = predict_scene(r)
    for k, c in cls.items():
        rows.append(dict(scene=r.scene, name=r.name, split=r.split, level=r.level, teacher=k, **fresh_scores(label, c)))
    a = agreement(label, cls["average"], raw)
    samples.append(sample_groups(a, refl, rng).assign(scene=r.name, split=r.split))
    dists[r.name] = fp_distances(label, a, r.px_m)
    dec = 4                                                  # every 4th pixel for the maps (10 m look)
    views[r.name] = dict(split=r.split, fc=(false_colour(refl[:, ::dec, ::dec]) * 255).astype(np.uint8),
                         agree=a[::dec, ::dec].copy(), px_m=r.px_m, dec=dec,
                         fn_blobs=biggest_blobs((a == 2)[::dec, ::dec]), fp_blobs=biggest_blobs((a == 3)[::dec, ::dec]),
                         crops={})
    for kind, centres in (("fn", views[r.name]["fn_blobs"]), ("fp", views[r.name]["fp_blobs"])):
        for (cr, cc) in centres[:1]:                         # one 512 x 512 crop at full resolution per kind
            r0 = int(np.clip(cr * dec - 256, 0, max(label.shape[0] - 512, 0)))
            c0 = int(np.clip(cc * dec - 256, 0, max(label.shape[1] - 512, 0)))
            views[r.name]["crops"][kind] = dict(fc=(false_colour(refl[:, r0:r0 + 512, c0:c0 + 512]) * 255).astype(np.uint8),
                                                raw=raw[r0:r0 + 512, c0:c0 + 512].copy(),
                                                agree=a[r0:r0 + 512, c0:c0 + 512].copy())
    print(f"  {r.name} ({r.split}) done, {(time.time() - t0) / 60:.1f} min")
    del label, raw, refl, cls, a

res = pd.DataFrame(rows)
spec = pd.concat(samples, ignore_index=True)
spec["ndvi"] = (spec.NIR - spec.R) / (spec.NIR + spec.R).clip(lower=1e-4)
spec["visible"] = spec[["B", "G", "R"]].mean(axis=1)
res.to_csv(f"{OUT}/scene_scores.csv", index=False)
spec.to_csv(f"{OUT}/pixel_samples.csv", index=False)
print(f"saved to {OUT}/")

## Per scene

Fresh-burn IoU of each teacher and of their average, then precision and recall of the average. Precision low means the teachers paint fresh burn the label doesn't have (blue in the maps); recall low means they leave out fresh burn the label has (amber).

In [ ]:
# [5] per scene: IoU of each teacher and the average; precision and recall of the average
wide = res.pivot_table(index=["split", "name", "level"], columns="teacher", values="iou").round(3)
avg = res[res.teacher == "average"].set_index(["split", "name", "level"])[["tp", "fp", "fn", "precision", "recall"]]
tab = wide.join(avg.round(3)).reset_index()
tab = tab.assign(o=tab.split.map(SPLIT_ORDER)).sort_values(["o", "name"]).drop(columns="o").reset_index(drop=True)
display(tab)

tot = res[res.teacher == "average"].groupby("split")[["tp", "fp", "fn"]].sum()
tot["iou"] = (tot.tp / (tot.tp + tot.fp + tot.fn)).round(3)
tot["precision"] = (tot.tp / (tot.tp + tot.fp)).round(3)
tot["recall"] = (tot.tp / (tot.tp + tot.fn)).round(3)
display(tot)

a = res[res.teacher == "average"].set_index("name").loc[tab.name]
fig, ax = plt.subplots(figsize=(13, 3.8))
xs = np.arange(len(a))
ax.bar(xs - 0.2, a.precision, 0.38, color=C_BLUE, label="precision (low: teachers paint extra fresh)")
ax.bar(xs + 0.2, a.recall, 0.38, color="#f2b134", label="recall (low: teachers leave labelled fresh out)")
ax.set_xticks(xs); ax.set_xticklabels([f"{n}\n{s}" for n, s in zip(a.index, a.split)], fontsize=7, rotation=30)
ax.set_ylim(0, 1); ax.axhline(0.8, color=INK2, lw=0.8, ls=":"); ax.legend(loc="lower right", fontsize=8)
ax.set_title("fresh burn, average of the teachers, per HEO scene", loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## What the disagreeing pixels look like

NDVI and visible brightness of four kinds of pixel, from up to 20,000 per scene: clear and fresh where teachers and labels agree, labelled fresh the teachers leave out (amber), and fresh the teachers add (blue). If the amber pixels look like the agreed clear ones, they are probably not visibly burnt and the label is the problem. If the blue pixels look like agreed fresh burn, they may be burns EFFIS doesn't have.

In [ ]:
# [6] NDVI and visible brightness of each kind of pixel, training scenes and held-out scenes apart
GROUP_NAMES = {0: "clear, agreed", 1: "fresh, agreed", 2: "labelled fresh,\nteachers no", 3: "teachers fresh,\nlabel no"}
GROUP_COLORS = [AGREE_COLORS[0], AGREE_COLORS[1], AGREE_COLORS[2], AGREE_COLORS[3]]
fig, ax = plt.subplots(2, 2, figsize=(14, 7.5))
for col, (title, sel) in enumerate((("training scenes (seen by the teachers)", spec.split == "train"),
                                    ("validation and test scenes", spec.split != "train"))):
    for row, (var, lab) in enumerate((("ndvi", "NDVI"), ("visible", "mean of blue, green, red (reflectance)"))):
        data = [spec.loc[sel & (spec.group == g), var].values for g in range(4)]
        keep = [i for i, d in enumerate(data) if len(d)]
        bp = ax[row, col].boxplot([data[i] for i in keep], showfliers=False, patch_artist=True, widths=0.6)
        for patch, i in zip(bp["boxes"], keep):
            patch.set_facecolor(GROUP_COLORS[i]); patch.set_edgecolor(INK2)
        ax[row, col].set_xticks(range(1, len(keep) + 1)); ax[row, col].set_xticklabels([GROUP_NAMES[i] for i in keep], fontsize=8)
        ax[row, col].set_ylabel(lab)
        if row == 0:
            ax[row, col].set_title(title, loc="left", fontsize=10)
plt.tight_layout(); plt.show()

med = spec.groupby(["split", "group"])[["ndvi", "visible", "NIR"]].median().round(3)
med.index = med.index.set_levels([med.index.levels[0], [GROUP_NAMES[g].replace("\n", " ") for g in med.index.levels[1]]])
display(med)

In [ ]:
# [7] how far the extra fresh (blue) is from a labelled burn: next to it = the edge, far = a separate object
bins, names = [0, 20, 100, 500, 1e9], ["< 20 m (edge)", "20-100 m", "100-500 m", "> 500 m (separate)"]
share = {}
for n, d in dists.items():
    if len(d):
        share[n] = np.histogram(d, bins=bins)[0] / len(d)
share = pd.DataFrame(share, index=names).T
share = share.loc[[n for n in tab.name if n in share.index]]
fig, ax = plt.subplots(figsize=(13, 3.6))
left = np.zeros(len(share))
for k, c in zip(names, ["#9cc3ee", "#5d9de0", C_BLUE, "#123f7a"]):
    ax.barh(range(len(share)), share[k], left=left, color=c, label=k)
    left += share[k].values
ax.set_yticks(range(len(share))); ax.set_yticklabels(share.index, fontsize=8); ax.invert_yaxis()
ax.set_xlim(0, 1); ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.0), fontsize=8)
ax.set_xlabel("share of the pixels the teachers call fresh and the label doesn't")
plt.tight_layout(); plt.show()

## Maps

Each scene whole (every 4th pixel, so a 10 m look): the image and where teachers and labels agree. Then, at full resolution, the biggest patch the teachers leave out and the biggest patch they add.

In [ ]:
# [8] every scene: image and agreement
legend = [Patch(facecolor=c, edgecolor="#8a8984", label=n) for n, c in zip(AGREE_NAMES, AGREE_COLORS)]
names_sorted = [n for n in tab.name if n in views]
fig, axes = plt.subplots(len(names_sorted), 2, figsize=(12, 6 * len(names_sorted)), squeeze=False)
for row, n in enumerate(names_sorted):
    v = views[n]
    s = res[(res.name == n) & (res.teacher == "average")].iloc[0]
    axes[row, 0].imshow(v["fc"])
    axes[row, 0].set_title(f"{n} ({v['split']})", fontsize=10, loc="left")
    axes[row, 1].imshow(v["agree"], cmap=AGREE_CMAP, vmin=-0.5, vmax=6.5, interpolation="nearest")
    axes[row, 1].set_title(f"IoU {s.iou:.2f}, precision {s.precision:.2f}, recall {s.recall:.2f}", fontsize=10, loc="left")
for a in axes.ravel():
    a.axis("off")
fig.legend(handles=legend, loc="lower center", ncol=4)
plt.tight_layout(rect=(0, 0.015, 1, 1)); plt.show()

In [ ]:
# [9] full resolution: the biggest patch the teachers leave out (amber) and the biggest they add (blue), per scene
items = [(n, k, v["crops"][k]) for n in names_sorted for k in ("fn", "fp") for v in [views[n]] if k in v["crops"]]
fig, axes = plt.subplots(len(items), 3, figsize=(13, 4.4 * len(items)), squeeze=False)
mask_cmap = ListedColormap([L.V3_COLORS[v] for v in L.V3_VALUES])
mask_norm = BoundaryNorm([-0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 200, 254.5, 255.5], mask_cmap.N)
for row, (n, k, c) in enumerate(items):
    axes[row, 0].imshow(c["fc"])
    axes[row, 0].set_title(f"{n}: biggest patch {'left out by' if k == 'fn' else 'added by'} the teachers", fontsize=9, loc="left")
    axes[row, 1].imshow(c["raw"], cmap=mask_cmap, norm=mask_norm, interpolation="nearest"); axes[row, 1].set_title("label", fontsize=9)
    axes[row, 2].imshow(c["agree"], cmap=AGREE_CMAP, vmin=-0.5, vmax=6.5, interpolation="nearest"); axes[row, 2].set_title("agreement", fontsize=9)
for a in axes.ravel():
    a.axis("off")
fig.legend(handles=legend, loc="lower center", ncol=4)
plt.tight_layout(rect=(0, 0.01, 1, 1)); plt.show()